# 06 · Covariance forecasts as hedging decisions

**RQ3:** Does a covariance estimate support a useful economic decision?
Keep one near contract and use the middle/far contracts as hedges. The objective
is reduced settlement risk, not an expected-return or trading-alpha strategy.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

from spde_covariance.empirical_research import load_research_evidence

evidence = load_research_evidence(Path("../experiments/results/shfe-research"))
print(evidence["protocol"]["status"])
print("Numerical source:", evidence["manifest"]["git_revision"])
print("Original baseline source:", evidence["protocol"]["baseline_source_revision"])
assert evidence["protocol"]["baseline_summary_sha256"]
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def table(rows, columns):
    print(" | ".join(columns))
    for row in rows:
        print(
            " | ".join(
                (
                    f"{row.get(c):.6g}" if isinstance(row.get(c), float) else str(row.get(c))
                    for c in columns
                )
            )
        )

## Decision problem and chronology

Given the issued PSD raw contract covariance $\widehat K_n$, solve

```math
\min_{h\in[-2,2]^2}(1,h^\top)\widehat K_n(1,h^\top)^\top.
```

The solver checks the unconstrained stationary point, edges and corners, so
singular PSD controls remain well-defined. It receives no future price. Each
rule selects its risk-model settings on 2023 validation QLIKE, then issues
positions using data through the origin. The same actual contracts are held
until next session. These are risk-selected policies, not separately
hedging-loss-tuned policies.

In [ ]:
joint = evidence["joint"]
print(joint["hedge_scope"])
print("Bounds:", joint["joint_config"]["hedge_bounds"])
table(
    joint["hedge_summary"],
    [
        "method",
        "common_count",
        "realized_rms_cny",
        "reduction_vs_unhedged",
        "predicted_rms_cny",
        "realized_over_predicted_second_moment",
    ],
)

In [ ]:
rows = joint["hedge_summary"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].bar(range(len(rows)), [r["realized_rms_cny"] for r in rows], color="0.4")
axes[0].set_xticks(range(len(rows)), [r["method"] for r in rows], rotation=75, fontsize=7)
axes[0].set(
    ylabel="Realized RMS settlement P&L (CNY)", title="One near contract plus bounded hedges"
)
hedged = [r for r in rows if r["method"] != "unhedged"]
axes[1].bar(
    range(len(hedged)), [r["mean_position_change_proxy"] or 0 for r in hedged], color="0.65"
)
axes[1].set_xticks(range(len(hedged)), [r["method"] for r in hedged], rotation=75, fontsize=7)
axes[1].set(
    ylabel="Mean absolute hedge-position change",
    title="Instability proxy; excludes roll/execution costs",
)
fig.tight_layout()
plt.show()

## Paired realized risk and position stability

All displayed rules use the common scored-origin cohort. Negative paired
squared-P&L differences favor the listed rule. Bootstrap blocks resample
paired sessions, not independent strategies. Mean hedge positions and bound
hits help detect unstable optimization even when average risk is small.

In [ ]:
table(
    joint["hedge_paired"],
    ["method", "reference", "count", "mean", "bootstrap_standard_error", "bootstrap_percentile_95"],
)
table(
    [r for r in joint["hedge_summary"] if r["method"] != "unhedged"],
    ["method", "mean_hedge_positions", "bound_hit_fraction", "mean_position_change_proxy"],
)
print(json.dumps(joint["availability"], indent=2))

## Interpretation and next decision

This closes the pipeline from data and curve reconstruction to model risk and
a constrained decision. It does not establish an executable backtest: contract
switching, actual fills, margin, roll mechanics, transaction costs and liquidity
capacity are not modeled. A later execution-aware study needs these inputs and
an uninspected evaluation period. Do not turn lower settlement RMS alone into
a profitability or CV performance claim.

## Reproduction and research boundary

This notebook reads checksummed aggregate evidence, not raw exchange files.
The numerical algorithms are in `spde_covariance`; use the
[protocol](../docs/empirical-research-protocol.md) to rerun local calibration.
Execute this notebook with `uv run python scripts/check_notebook.py notebooks/06_hedging_decisions.ipynb`.

The original historical benchmark remains fixed. New analyses on the inspected
2024–2025 period are exploratory, even though forecast issuance is past-only.
No final untouched confirmation period, executable trading backtest or physical
forcing identification is claimed. See the [roadmap](../README.md).